# PoC step 1: build the synthetic dataset
Runs `make_poc_pairs.py` on the real `MatchedPoolReference.csv`, checks the output, and creates the SQLite database from `schema.sql`.

**Before you start:** put `make_poc_pairs.py`, `participants.yaml`, and `schema.sql` in `bw-interop-poc/src` in Google Drive, and `MatchedPoolReference.csv` in `bw-interop-poc/data`. Cell 2 can upload the three small files for you.

In [5]:
# Cell 1: mount Drive and set paths
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/bw-interop-poc'
for d in ['src', 'data', 'docs']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)
POOL = f'{BASE}/data/MatchedPoolReference.csv'
print('Base folder:', BASE)
print('src :', os.listdir(f'{BASE}/src'))
print('data:', os.listdir(f'{BASE}/data'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Base folder: /content/drive/MyDrive/bw-interop-poc
src : ['make_poc_pairs.py', 'participants.yaml', 'schema.sql']
data: ['Matched_Pool_Reference.csv']


In [6]:
# Cell 2 (optional): upload the three small files if they are not in Drive yet
# Choose make_poc_pairs.py, participants.yaml and schema.sql when the file picker opens.
from google.colab import files
import shutil
up = files.upload()
for name in up:
    shutil.move(name, f'{BASE}/src/{name}')
print(os.listdir(f'{BASE}/src'))

Saving make_poc_pairs.py to make_poc_pairs.py
Saving participants.yaml to participants.yaml
Saving schema.sql to schema.sql
['make_poc_pairs.py', 'participants.yaml', 'schema.sql']


In [7]:
# Cell 3: check the matched pool before running anything
import pandas as pd
print('Pool file found:', os.path.exists(POOL))
head = pd.read_csv(POOL, nrows=3)
print('Columns:', head.columns.tolist())
need = {'step', 'amount', 'nameOrig', 'nameDest'}
print('Missing columns:', need - set(head.columns))   # must print set()
n_rows = sum(1 for _ in open(POOL)) - 1
print('Rows in pool:', n_rows)                         # about 953,921 expected

Pool file found: True
Columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']
Missing columns: set()
Rows in pool: 953921


In [8]:
# Cell 4: install the one extra dependency and run the generator
!pip -q install pyyaml
!python {BASE}/src/make_poc_pairs.py --matched-pool {POOL} --participants {BASE}/src/participants.yaml --n 400 --seed 42 --out {BASE}/data/poc

pacs.008 records: 402  Orange records: 394  manifest rows: 408
expected_outcome
MATCHED       352
DISCREPANT     24
REJECTED       20
UNMATCHED      12
anomaly_type
(clean)             336
DUPLICATE_E2E        16
AMOUNT_MISMATCH      12
FAILED_STATUS        12
LAG_OUTLIER           8
MISSING_CURRENCY      6
ORPHAN_OM             6
ORPHAN_ISO            6
UNKNOWN_BIC           6


In [9]:
# Cell 5: verify the output (stops with an error if a count is wrong)
import json
import pandas as pd
from xml.etree import ElementTree as ET
NS = {'p': 'urn:iso:std:iso:20022:tech:xsd:pacs.008.001.08'}
root = ET.parse(f'{BASE}/data/poc_pacs008.xml').getroot()
iso_n = len(root.findall('.//p:CdtTrfTxInf', NS))
om_n = len(json.load(open(f'{BASE}/data/poc_orange.json')))
man = pd.read_csv(f'{BASE}/data/poc_manifest.csv', dtype=str).fillna('')
print('pacs.008 records:', iso_n, '| Orange records:', om_n, '| manifest rows:', len(man))
assert (iso_n, om_n, len(man)) == (402, 394, 408), 'Counts differ from the expected 402 / 394 / 408 for n=400'
print(man['expected_outcome'].value_counts().to_string())
print(man['anomaly_type'].replace('', '(clean)').value_counts().to_string())
man.head()

pacs.008 records: 402 | Orange records: 394 | manifest rows: 408
expected_outcome
MATCHED       352
DISCREPANT     24
REJECTED       20
UNMATCHED      12
anomaly_type
(clean)             336
DUPLICATE_E2E        16
AMOUNT_MISMATCH      12
FAILED_STATUS        12
LAG_OUTLIER           8
MISSING_CURRENCY      6
ORPHAN_OM             6
ORPHAN_ISO            6
UNKNOWN_BIC           6


,pair_id,end_to_end_id,anomaly_type,iso_present,om_present,expected_outcome,expected_flag,amount_iso,amount_om,currency_iso,creditor_msisdn,debtor_account,debtor_bic,creditor_bic,iso_settlement_date,om_created_at,lag_days
0,P0001,E2E-BW-20260807-0001,,1,1,MATCHED,,18521.71,18521.71,BWP,26779399039,1155104816,MOCKBWBRXXX,MOCKBWOMXXX,2026-08-08,2026-08-07T19:02:00Z,1
1,P0002,E2E-BW-20260830-0002,,1,1,MATCHED,,25257.34,25257.34,BWP,26772119393,1703697384,MOCKBWBSXXX,MOCKBWOMXXX,2026-09-02,2026-08-30T14:17:00Z,3
2,P0003,E2E-BW-20260810-0003,,1,1,MATCHED,,28661.04,28661.04,BWP,26771188152,2325609883,MOCKBWBGXXX,MOCKBWOMXXX,2026-08-10,2026-08-10T13:44:00Z,0
3,P0004,E2E-BW-20260816-0004,MISSING_CURRENCY,1,1,REJECTED,,3872.00,3872.00,,26772258352,2478283455,MOCKBWBSXXX,MOCKBWOMXXX,2026-08-17,2026-08-16T11:17:00Z,1
4,P0005,E2E-BW-20260809-0005,,1,1,MATCHED,,5536.83,5536.83,BWP,26771827415,9827557155,MOCKBWSCXXX,MOCKBWOMXXX,2026-08-11,2026-08-09T11:51:00Z,2


In [10]:
# Cell 6: create the SQLite database from schema.sql and seed the participants
import sqlite3, yaml
DB = f'{BASE}/data/poc.db'
if os.path.exists(DB):
    os.remove(DB)
db = sqlite3.connect(DB)
db.executescript(open(f'{BASE}/src/schema.sql').read())
for p in yaml.safe_load(open(f'{BASE}/src/participants.yaml'))['participants']:
    db.execute('INSERT INTO participants(code,name,type,bic,active,note) VALUES (?,?,?,?,?,?)',
               (p['code'], p['name'], p['type'], p['bic'], int(p['active']), p.get('note')))
db.commit()
print('Tables:', [r[0] for r in db.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")])
print('Participants:', db.execute('SELECT COUNT(*) FROM participants').fetchone()[0])
print('Mapping registry rows:', db.execute('SELECT COUNT(*) FROM mapping_registry').fetchone()[0])

Tables: ['exceptions', 'load_audit', 'mapping_registry', 'participants', 'parties', 'reconciliation_records', 'settlement_journal', 'sqlite_sequence', 'status_reports', 'transaction_source_ids', 'transactions']
Participants: 10
Mapping registry rows: 30


## Done when
- Cell 3 prints `Missing columns: set()`.
- Cell 5 prints `402 | 394 | 408` and does not raise an error.
- Cell 6 lists 10 tables, 10 participants, and 30 mapping rows.

Copy the printed output of cells 3, 5, and 6 into the decision log and share it with the team.